In [1]:
# Install necessary packages if not already installed
import Pkg
Pkg.add(["CSV", "DataFrames", "Flux", "Statistics", "Random", "Dates", "StatsBase"])

using CSV
using DataFrames
using Flux
using Statistics
using Random
using Dates
using StatsBase

# Include the provided utility functions
# This file contains: buildClassANN, trainClassANN, crossvalidation, ANNCrossValidation, etc.
include("utils.jl")

println("Environment setup complete. Utils loaded.")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


Environment setup complete. Utils loaded.


In [2]:
# Load the dataset
const DATA_PATH = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"

println("Loading data...")
df = CSV.read(DATA_PATH, DataFrame)

# --- SUBSAMPLING (Keep enabled for testing) ---
const DO_SUBSAMPLE = true 
const SUBSAMPLE_SIZE = 10000 

if DO_SUBSAMPLE
    println("\n⚠️ SUBSAMPLING ENABLED: Using only $SUBSAMPLE_SIZE rows.")
    df = df[1:SUBSAMPLE_SIZE, :]
end

# CORRECTED TARGET based on your image
target_col = "Is Fraudulent"

println("Data loaded. Rows: ", size(df, 1))
println("Target Column: ", target_col)
println("Columns found: ", names(df))

Loading data...

⚠️ SUBSAMPLING ENABLED: Using only 10000 rows.
Data loaded. Rows: 10000
Target Column: Is Fraudulent
Columns found: ["Transaction ID", "Customer ID", "Transaction Amount", "Transaction Date", "Payment Method", "Product Category", "Quantity", "Customer Age", "Customer Location", "Device Used", "IP Address", "Shipping Address", "Billing Address", "Is Fraudulent", "Account Age Days", "Transaction Hour"]


In [3]:
function preprocess_data(dataframe)
    data = copy(dataframe)
    
    # 1. Handle Dates
    # Your screenshot shows 'Transaction Hour' might already exist.
    # If it exists, we use it. If not, we try to create it from 'Transaction Date'.
    if "Transaction Hour" in names(data)
        println("Using existing 'Transaction Hour' column.")
    elseif "Transaction Date" in names(data)
        println("Extracting Hour from 'Transaction Date'...")
        try
            data.Transaction_Hour = [Hour(DateTime(t, "yyyy-mm-dd HH:MM:SS")).value for t in data[!, "Transaction Date"]]
        catch
            println("⚠️ Date parsing warning. Using 0 for hour.")
            data.Transaction_Hour = zeros(nrow(data))
        end
    end

    # 2. Remove columns that are Strings/IDs and not useful for the ANN
    # We drop "Customer Location" because city names are text and will crash the matrix conversion.
    cols_to_drop = [
        "Transaction ID", 
        "Customer ID", 
        "Transaction Date", # We rely on Transaction Hour/Account Age instead
        "IP Address", 
        "Shipping Address", 
        "Billing Address", 
        "Customer Location" 
    ]
    
    # Only drop columns that actually exist in the dataframe to avoid errors
    actual_cols_to_drop = intersect(names(data), cols_to_drop)
    select!(data, Not(actual_cols_to_drop))

    # 3. Encode Categorical Variables to Integers
    # These are the exact names from your screenshots
    categorical_cols = ["Payment Method", "Product Category", "Device Used"]
    
    for col in categorical_cols
        if col in names(data)
            println("Encoding column: $col")
            # Create a mapping Dict: String -> Int
            unique_vals = unique(data[!, col])
            mapping = Dict(val => i for (i, val) in enumerate(unique_vals))
            data[!, col] = [mapping[x] for x in data[!, col]]
        end
    end

    return data
end

println("Preprocessing data...")
df_processed = preprocess_data(df)

# Verification: Check if any String columns remain
types = eltype.(eachcol(df_processed))
if any(x -> x <: AbstractString, types)
    println("\n❌ WARNING: Text columns still exist! Matrix conversion will fail.")
    # Print which columns are causing the issue
    for (name, type) in zip(names(df_processed), types)
        if type <: AbstractString
            println("  - BAD COLUMN: $name (Type: $type)")
        end
    end
else
    println("\n✅ Data is clean. All columns are numeric.")
    println("Remaining Features: ", names(df_processed))
end

Preprocessing data...
Using existing 'Transaction Hour' column.
Encoding column: Payment Method
Encoding column: Product Category
Encoding column: Device Used

✅ Data is clean. All columns are numeric.
Remaining Features: ["Transaction Amount", "Payment Method", "Product Category", "Quantity", "Customer Age", "Device Used", "Is Fraudulent", "Account Age Days", "Transaction Hour"]


In [4]:
# Define the target column again to be safe
target_column = "Is Fraudulent"

# Select input features (all columns except the target)
input_cols = setdiff(names(df_processed), [target_column])

println("Preparing Matrix...")

try
    # 1. Input Matrix (Features)
    global inputs = Matrix{Float32}(df_processed[:, input_cols])
    
    # 2. Target Vector (Labels)
    global targets = df_processed[:, target_column]

    println("✅ Success!")
    println("Input Matrix Dimensions: ", size(inputs))
    println("Target Vector Dimensions: ", size(targets))
    println("Data Type: ", eltype(inputs))
    
catch e
    println("\n❌ ERROR during Matrix conversion.")
    println("Check the output of Cell 3 again.")
    rethrow(e)
end

Preparing Matrix...
✅ Success!
Input Matrix Dimensions: (10000, 8)
Target Vector Dimensions: (10000,)
Data Type: Float32


In [15]:
# --- Hyperparameters ---

# k-fold Cross Validation (Standard is 10) 
# If subsampling or testing, you might reduce k to 5 to save time.
k_folds = 10 

# ANN Topology [cite: 29]
# Input layer size is automatic. 
# We define hidden layers. Example: [16, 8] neurons.
topology = [64, 32] 

# Activation functions for hidden layers
# Usually sigmoid (σ) or relu. utils.jl defaults to σ.
transferFunctions = fill(σ, length(topology))

# Training Parameters
# Important: Since we use 'ANNCrossValidation', we deal with multiple executions per fold.
numExecutions = 10 # Reduced from 50 to 10 for reasonable runtime on large data.
maxEpochs = 500
learningRate = 0.01
validationRatio = 0.2 # Use 20% of training data for validation (Early Stopping)
maxEpochsVal = 25     # Stop if validation doesn't improve for 10 epochs

println("Configuration Set:")
println("Topology: Inputs -> $topology -> Outputs")
println("Folds: $k_folds | Executions per fold: $numExecutions")

Configuration Set:
Topology: Inputs -> [64, 32] -> Outputs
Folds: 10 | Executions per fold: 10


In [16]:
# --- BALANCING THE DATA (OVERSAMPLING) ---
# Since utils.jl does not support weighted loss, we must balance the input data.

println("Balancing dataset via Oversampling...")

# 1. Separate indices
idx_fraud = findall(x -> x == 1, df_processed[!, target_column])
idx_legit = findall(x -> x == 0, df_processed[!, target_column])

# 2. Calculate how many frauds we need to add
num_legit = length(idx_legit)
num_fraud = length(idx_fraud)
num_to_add = num_legit - num_fraud

println("Original: $num_legit Legit, $num_fraud Fraud")

# 3. Randomly sample fraud indices to duplicate
extra_fraud_indices = rand(idx_fraud, num_to_add)

# 4. Create new balanced dataset indices
balanced_indices = vcat(idx_legit, idx_fraud, extra_fraud_indices)
shuffle!(balanced_indices)

# 5. Create the new Inputs/Targets Matrices
# IMPORTANT: We overwrite 'inputs' and 'targets' so the next cell uses the balanced version
global inputs = Matrix{Float32}(df_processed[balanced_indices, input_cols])
global targets = df_processed[balanced_indices, target_column]

# 6. RE-GENERATE CV INDICES for the new balanced data
# We must re-run the crossvalidation function because the data size changed!
global cv_indices = crossvalidation(vec(targets .== 1), k_folds)

println("✅ Data Balanced!")
println("New Dataset Size: ", length(targets))
println("New Class Distribution: ", countmap(targets))

Balancing dataset via Oversampling...
Original: 9525 Legit, 475 Fraud
✅ Data Balanced!
New Dataset Size: 19050
New Class Distribution: Dict(0 => 9525, 1 => 9525)


In [17]:
# Generate Stratified Cross-Validation Indices
println("Generating stratified cross-validation indices...")

# FIX: Convert targets to a Boolean Vector (true/false)
# This ensures utils.jl uses the correct binary classification logic
# instead of getting confused and thinking it's a matrix.
targets_bool = vec(targets .== 1)

# Now pass the boolean vector to the function
cv_indices = crossvalidation(targets_bool, k_folds)

println("Indices generated successfully.")
println("First 20 indices: ", cv_indices[1:20])

Generating stratified cross-validation indices...
Indices generated successfully.
First 20 indices: [4, 1, 2, 1, 7, 8, 3, 5, 5, 10, 5, 7, 3, 10, 4, 2, 9, 2, 3, 1]


In [18]:
println("Starting ANN Cross-Validation...")
println("This may take time depending on dataset size and hardware.")

# Call the function from utils.jl 
# Returns a tuple of tuples: ((acc_mean, acc_std), ..., confusionMatrix)
results = ANNCrossValidation(
    topology,
    (inputs, targets),
    cv_indices;
    numExecutions = numExecutions,
    transferFunctions = transferFunctions,
    maxEpochs = maxEpochs,
    learningRate = learningRate,
    validationRatio = validationRatio,
    maxEpochsVal = maxEpochsVal
)

println("Training Complete!")

Starting ANN Cross-Validation...
This may take time depending on dataset size and hardware.

Fold 1/10
  Execution 10/10

Fold 2/10
  Execution 10/10

Fold 3/10
  Execution 10/10

Fold 4/10
  Execution 10/10

Fold 5/10
  Execution 10/10

Fold 6/10
  Execution 10/10

Fold 7/10
  Execution 10/10

Fold 8/10
  Execution 10/10

Fold 9/10
  Execution 10/10

Fold 10/10
  Execution 10/10
Training Complete!


In [19]:
# Unpack the results [cite: 64]
(acc, err, sens, spec, ppv, npv, f1, global_cm) = results

println("=== Cross-Validation Results (Mean ± STD) ===")
println("Accuracy:    $(round(acc[1]*100, digits=2))% ± $(round(acc[2]*100, digits=2))%")
println("Error Rate:  $(round(err[1]*100, digits=2))% ± $(round(err[2]*100, digits=2))%")
println("Sensitivity: $(round(sens[1]*100, digits=2))% ± $(round(sens[2]*100, digits=2))%")
println("Specificity: $(round(spec[1]*100, digits=2))% ± $(round(spec[2]*100, digits=2))%")
println("Precision (PPV): $(round(ppv[1]*100, digits=2))% ± $(round(ppv[2]*100, digits=2))%")
println("F1 Score:    $(round(f1[1]*100, digits=2))% ± $(round(f1[2]*100, digits=2))%")

println("\n=== Global Confusion Matrix (Summed & Averaged) ===")
# Note: Values are floats because they are averages of multiple executions
display(global_cm)

# Interpretation Helper
println("\n--- Matrix Interpretation ---")
println("Row 1: Actual Non-Fraud")
println("Row 2: Actual Fraud")
println("Col 1: Predicted Non-Fraud")
println("Col 2: Predicted Fraud")

TP = global_cm[2,2]
FP = global_cm[1,2]
FN = global_cm[2,1]

println("\nFraud Detection Analysis:")
println("We successfully caught approx $(round(TP, digits=1)) fraud cases.")
println("We missed approx $(round(FN, digits=1)) fraud cases.")
println("We falsely accused approx $(round(FP, digits=1)) legitimate users.")

=== Cross-Validation Results (Mean ± STD) ===
Accuracy:    74.64% ± 0.94%
Error Rate:  25.36% ± 0.94%
Sensitivity: 71.22% ± 1.88%
Specificity: 78.07% ± 1.15%
Precision (PPV): 76.47% ± 0.91%
F1 Score:    73.73% ± 1.18%

=== Global Confusion Matrix (Summed & Averaged) ===


2×2 Matrix{Float64}:
 7436.0  2089.0
 2741.6  6783.4


--- Matrix Interpretation ---
Row 1: Actual Non-Fraud
Row 2: Actual Fraud
Col 1: Predicted Non-Fraud
Col 2: Predicted Fraud

Fraud Detection Analysis:
We successfully caught approx 6783.4 fraud cases.
We missed approx 2741.6 fraud cases.
We falsely accused approx 2089.0 legitimate users.


In [20]:
# --- CELLA 9: Analisi del Livello di Rischio (Risk Scoring) ---

println("Generazione del Profilo di Rischio...")

# 1. Addestriamo un modello finale su un split 80/20 (Hold-Out) per analizzare le probabilità
# Usiamo la topologia "vincente" [16, 8]
(train_idx, val_idx, test_idx) = holdOut(size(inputs, 1); Pval=0.0, Ptest=0.2)

# Addestramento rapido
(ann_final, _) = trainClassANN(topology, 
    (inputs[train_idx, :], targets[train_idx, :]);
    maxEpochs=500, learningRate=0.01, printLoss=false
)

# 2. Otteniamo le probabilità pure (valori tra 0 e 1) sul set di test
# Trasponiamo inputs per Flux (features x samples)
test_probs = ann_final(inputs[test_idx, :]')' 

# 3. Categorizziamo il rischio in fasce
risk_levels = String[]
for p in test_probs
    if p < 0.2
        push!(risk_levels, "1. Basso Rischio")
    elseif p < 0.6
        push!(risk_levels, "2. Rischio Moderato")
    elseif p < 0.9
        push!(risk_levels, "3. Alto Rischio")
    else
        push!(risk_levels, "4. Rischio Critico (Frode Certa)")
    end
end

# 4. Visualizziamo la distribuzione
println("\n--- Distribuzione dei Livelli di Rischio (Test Set) ---")
risk_counts = countmap(risk_levels)
# Ordiniamo per chiave per una lettura pulita
for level in sort(collect(keys(risk_counts)))
    count = risk_counts[level]
    perc = round(count / length(test_probs) * 100, digits=1)
    println("$level: $count transazioni ($perc%)")
end

Generazione del Profilo di Rischio...


LoadError: MethodError: no method matching trainClassANN(::Vector{Int64}, ::Tuple{Matrix{Float32}, Matrix{Int64}}; maxEpochs::Int64, learningRate::Float64, printLoss::Bool)

[0mClosest candidates are:
[0m  trainClassANN(::AbstractVector{<:Int64}, [91m::Tuple{AbstractMatrix{<:Real}, AbstractMatrix{Bool}}[39m; validationDataset, transferFunctions, maxEpochs, minLoss, learningRate, maxEpochsVal)[91m got unsupported keyword argument "printLoss"[39m
[0m[90m   @[39m [35mMain[39m [90m~/Machine Learning/Project/[39m[90m[4mutils.jl:235[24m[39m


In [21]:
# --- CELLA 10: Ranking Frodi per Frequenza e Costo ---

using DataFrames

println("\n--- RANKING FRODI: Dove perdiamo più soldi? ---")

# Creiamo un DataFrame temporaneo con i dati originali (prima della normalizzazione)
# Nota: Usiamo il dataframe 'df_processed' che contiene i dati leggibili
# Filtriamo solo le righe che sono REALMENTE frodi (Is Fraudulent == 1)
fraud_data = filter(row -> row["Is Fraudulent"] == 1, df_processed)

# 1. Ranking per CATEGORIA DI PRODOTTO
# Raggruppiamo per categoria e calcoliamo: Conteggio e Somma degli importi
cat_stats = combine(groupby(fraud_data, "Product Category"), 
                    nrow => :Count, 
                    "Transaction Amount" => sum => :Total_Loss)

# Ordiniamo per Perdita Totale (dal più costoso al meno costoso)
sort!(cat_stats, :Total_Loss, rev=true)

println("\nTOP 5 Categorie per Costo Frode (Denaro Perso):")
display(first(cat_stats, 5))

# 2. Ranking per DEVICE (Dispositivo)
dev_stats = combine(groupby(fraud_data, "Device Used"), 
                    nrow => :Count,
                    "Transaction Amount" => mean => :Avg_Loss)

sort!(dev_stats, :Count, rev=true)
println("\nTOP Device utilizzati per frodare (Frequenza):")
display(dev_stats)

println("\nInsight: Il ranking sopra mostra dove dovremmo concentrare i controlli di sicurezza.")


--- RANKING FRODI: Dove perdiamo più soldi? ---

TOP 5 Categorie per Costo Frode (Denaro Perso):


Row,Product Category,Count,Total_Loss
,Int64,Int64,Float64
1,4,99,64000.0
2,1,90,58917.7
3,2,96,54321.5
4,5,98,53819.0
5,3,92,43633.8



TOP Device utilizzati per frodare (Frequenza):


Row,Device Used,Count,Avg_Loss
,Int64,Int64,Float64
1,1,160,565.673
2,3,160,598.068
3,2,155,570.925



Insight: Il ranking sopra mostra dove dovremmo concentrare i controlli di sicurezza.


In [22]:
# --- CELLA 11: Analisi Comportamentale (Orari e Anomalie) ---

println("\n--- ANALISI COMPORTAMENTO ANOMALO ---")

# 1. Analisi Temporale: A che ora avvengono le frodi?
# Raggruppiamo le frodi per "Transaction Hour"
hour_stats = combine(groupby(fraud_data, "Transaction Hour"), nrow => :Count)
sort!(hour_stats, :Count, rev=true)

println("Orari più pericolosi (Top 3 ore con più frodi):")
for row in eachrow(first(hour_stats, 3))
    println("  Ore $(row["Transaction Hour"]): $(row[:Count]) frodi rilevate")
end

# 2. Anomalie sugli Importi (High Value Whales)
mean_val = mean(df_processed[:, "Transaction Amount"])
fraud_mean = mean(fraud_data[:, "Transaction Amount"])

println("\nConfronto Importi Medi:")
println("  Importo medio transazione legittima (stima): $(round(mean_val, digits=2))")
println("  Importo medio transazione fraudolenta:       $(round(fraud_mean, digits=2))")

if fraud_mean > mean_val
    println(">> CONCLUSIONE: Le frodi tendono ad avere importi PIÙ ALTI della media.")
else
    println(">> CONCLUSIONE: Le frodi tendono ad essere piccoli importi (micro-frodi).")
end


--- ANALISI COMPORTAMENTO ANOMALO ---
Orari più pericolosi (Top 3 ore con più frodi):
  Ore 5: 51 frodi rilevate
  Ore 1: 49 frodi rilevate
  Ore 2: 45 frodi rilevate

Confronto Importi Medi:
  Importo medio transazione legittima (stima): 226.09
  Importo medio transazione fraudolenta:       578.3
>> CONCLUSIONE: Le frodi tendono ad avere importi PIÙ ALTI della media.
